# AI News Agents — model-change tracker + legal subagent

This notebook demonstrates an agent that watches **AI provider web pages** for model changes (launches, deprecations, pricing, API changes) and a **legal subagent** that watches terms of service, usage policies and EU AI Act pages affecting LLM providers. Both produce a cited digest saved as Markdown (emailed later).

## Design: deterministic pipeline, LLM only where judgment is needed

```
 pages ─► 1 Fetch ─► 2 Normalize ─► 3 Change checks ──────────────► 4 Triage (cheap LLM)
          ETag/304                   hash · new lines · text ratio      only when the checks
                                     DOM ratio · keywords               can't decide
                                                                          │ meaningful?
                                                                          ▼
            8 Save    ◄─ 7 Digest (LLM) ◄─ 6 Dedupe ◄─ 5 Extract structured events (LLM)
                                                         + verbatim-quote guard
```

## Setup

Auth uses Google Application Default Credentials — run once in a terminal, then set the values in the config cell below:

```
gcloud auth application-default login
```

In [ ]:
# %pip install -r requirements.txt

# All configuration lives here. Edit these values directly.

# Vertex AI
PROJECT_ID = "your-gcp-project"
REGION = "global"

# Gemini models, swappable per step.
# Check Vertex Model Garden for the model versions enabled in your project.
TRIAGE_MODEL = "gemini-2.5-flash-lite"
EXTRACT_MODEL = "gemini-2.5-pro"
DIGEST_MODEL = "gemini-2.5-pro"

# USD per 1M tokens (input, output), for the cost estimates in section 11. Thinking tokens count as output.
# Check current prices: https://cloud.google.com/vertex-ai/generative-ai/pricing (Pro: prompts up to 200k tokens)
MODEL_PRICES = {
    "gemini-2.5-flash-lite": (0.10, 0.40),
    "gemini-2.5-flash": (0.30, 2.50),
    "gemini-2.5-pro": (1.25, 10.00),
}

# HTTP
HTTP_TIMEOUT_SECONDS = 30
USER_AGENT = "ai-news-agents-demo/0.1 (change monitoring)"

# Paths and limits
SNAPSHOT_DIR = "snapshots"
BASELINE_DIR = "baselines"   # seeded older copies, used when a source has no snapshot yet
OUTPUT_DIR = "output"
TRIAGE_LOG = "output/triage_log.jsonl"   # every model verdict + its signals: training data for a future classifier
MAX_DIFF_CHARS = 20_000

# Change checks (section 5). Cleaning strips these patterns from every line before comparing.
# Keep them narrow: a pattern that matches real content hides real changes.
IGNORE_PATTERNS = [
    # "Last updated: October 28, 2026" / "Updated 2026-10-28" (date-only lines)
    r"^(last )?(updated|modified|reviewed)( on)?:?\s+(\w+ \d{1,2},? \d{4}|\d{1,2} \w+ \d{4}|\d{4}-\d{2}-\d{2})\.?$",
    r"\b\d+\s*min(ute)?s? read\b",                                         # "5 min read"
    r"^(©|copyright\s*©?)\s*(\d{4}\s*[-–]\s*)?\d{4}\b.*$",                   # "© 2026 Example Inc."
    r"^(?=.{0,200}$)(?=.*\bcookies?\b)(?=.*\b(accept|reject|consent|preferences|settings)\b).*$",  # cookie banners
    r"^(was this page helpful\??|share|print|back to top|skip to (main )?content|on this page)$",
]
BROKEN_RATIO = 0.3   # cleaned text shrank below 30% of the previous copy: extraction probably broke
DRASTIC_TEXT = 0.5   # more than half the text changed...
DRASTIC_DOM = 0.5    # ...and more than half the page's tag structure changed: likely a redesign

# Savings estimate (section 11) when a run made no triage calls to measure
FALLBACK_TRIAGE_SECONDS = 2.0
FALLBACK_TRIAGE_USD = 0.0005

## Sources

Primary sources only: provider docs, changelogs, pricing and policy pages, and official EU pages. Every source is a web page and goes through the same checks.

In [ ]:
MODEL_SOURCES = [
    {"name": "Anthropic model deprecations", "provider": "Anthropic",
     "url": "https://docs.anthropic.com/en/docs/about-claude/model-deprecations"},
    {"name": "Anthropic API release notes", "provider": "Anthropic",
     "url": "https://docs.anthropic.com/en/release-notes/api"},
    {"name": "Gemini API changelog", "provider": "Google",
     "url": "https://ai.google.dev/gemini-api/docs/changelog"},
    {"name": "Google AI Studio pricing", "provider": "Google",
     "url": "https://aistudio.google.com/docs/pricing"},
    {"name": "Vertex AI generative AI release notes", "provider": "Google",
     "url": "https://cloud.google.com/vertex-ai/generative-ai/docs/release-notes"},
    {"name": "Mistral changelog", "provider": "Mistral",
     "url": "https://docs.mistral.ai/getting-started/changelog/"},
]

LEGAL_SOURCES = [
    {"name": "Anthropic usage policy", "provider": "Anthropic",
     "url": "https://www.anthropic.com/legal/aup"},
    {"name": "OpenAI usage policies", "provider": "OpenAI",
     "url": "https://openai.com/policies/usage-policies/"},
    {"name": "EU AI Act regulatory framework", "provider": "European Commission",
     "url": "https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai"},
    {"name": "EU GPAI Code of Practice", "provider": "European Commission",
     "url": "https://digital-strategy.ec.europa.eu/en/policies/contents-code-gpai"},
]

## Event schemas

The LLM must return these shapes (structured output), so downstream code never parses free text.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class Triage(BaseModel):
    meaningful: bool = Field(description="False for layout, navigation, typo or boilerplate-only changes")
    category: str
    severity: Literal["low", "medium", "high"]
    reason: str


class ChangeEvent(BaseModel):
    provider: str
    model: str | None
    change_type: Literal["new_model", "deprecation", "retirement", "pricing", "api_change",
                         "rate_limit", "capability", "other"]
    effective_date: str | None = Field(description="ISO date if the page states one")
    summary: str
    quote: str = Field(description="Verbatim sentence or table row copied from the NEW page text")
    source_url: str


class LegalEvent(BaseModel):
    jurisdiction: Literal["EU", "US-federal", "US-state", "UK", "other"]
    instrument: Literal["legislation", "regulation", "guidance", "terms_of_service",
                        "litigation", "enforcement", "other"]
    title: str
    parties: list[str] = Field(description="Companies, agencies or courts involved")
    effective_date: str | None
    summary: str
    impact: str = Field(description="Practical effect on companies providing LLM services")
    quote: str = Field(description="Verbatim sentence copied from the NEW page text")
    source_url: str


class ChangeEvents(BaseModel):
    events: list[ChangeEvent]


class LegalEvents(BaseModel):
    events: list[LegalEvent]

## Fetch + normalize

HTML → main text (trafilatura strips nav and footers, keeps tables). JavaScript-app pages that ship their content as text in `<noscript><pre>` (such as the AI Studio docs) use that copy instead, because it keeps its line breaks. `fetch()` sends the `ETag` / `Last-Modified` values saved by the previous run, so an unchanged page can answer `304 Not Modified` without being downloaded.

In [ ]:
import time
from dataclasses import dataclass

import httpx
import lxml.etree
import lxml.html
import trafilatura

HTTP = httpx.Client(follow_redirects=True, timeout=HTTP_TIMEOUT_SECONDS, headers={"User-Agent": USER_AGENT})


@dataclass
class Fetched:
    html: str | None          # None when the server answered 304 Not Modified
    etag: str | None
    last_modified: str | None
    seconds: float


def fetch(url: str, validators: dict | None = None) -> Fetched:
    validators = validators or {}
    headers = {}
    if validators.get("etag"):
        headers["If-None-Match"] = validators["etag"]
    if validators.get("last_modified"):
        headers["If-Modified-Since"] = validators["last_modified"]
    started = time.perf_counter()
    resp = HTTP.get(url, headers=headers)
    seconds = time.perf_counter() - started
    if resp.status_code == 304:
        return Fetched(None, validators.get("etag"), validators.get("last_modified"), seconds)
    resp.raise_for_status()
    return Fetched(resp.text, resp.headers.get("etag"), resp.headers.get("last-modified"), seconds)


def _noscript_text(html: str) -> str:
    """JS-app pages (e.g. AI Studio docs) ship their content as source text in <noscript><pre>."""
    try:
        root = lxml.html.fromstring(html)
    except (lxml.etree.ParserError, ValueError):
        return ""
    return "\n".join(pre.text_content() for pre in root.xpath("//noscript//pre"))


def normalize(html: str) -> str:
    text = trafilatura.extract(html, include_tables=True, include_links=False, favor_recall=True) or ""
    # trafilatura flattens <pre> line breaks, so prefer the no-JS copy when it holds more of the page
    fallback = _noscript_text(html)
    return (fallback if len(fallback) > len(text) else text).strip()


demo_src = MODEL_SOURCES[0]
try:
    demo_page = fetch(demo_src["url"])
    demo_text = normalize(demo_page.html)
    print(f"{demo_src['name']}: {len(demo_text):,} chars in {demo_page.seconds:.2f}s (ETag: {demo_page.etag})\n")
    print(demo_text[:1500])
except httpx.HTTPError as e:
    print(f"Could not fetch {demo_src['url']}: {e}")

## Snapshots + change checks

Each source keeps three files in `snapshots/<domain>/`: the normalized text (`.txt`), the raw HTML (`.html`, for
the DOM check) and the HTTP validators (`.meta.json`, for the next `304`). When a source has no snapshot yet, the run
compares against its **seeded baseline** in `baselines/<domain>/<source>.md`: the source's text with its newest entries
removed, so the first run reports real, recent changes. Baselines are text only, so the DOM check is skipped that time.

Every page goes through all of these checks before any model is called:

| Check | Outcome | What happens |
|---|---|---|
| `304 Not Modified`, or the cleaned texts hash the same | `none` | no change |
| Cleaned text shrank below `BROKEN_RATIO` of the previous copy | `broken` | flagged; the previous snapshot is kept; no LLM |
| No new lines (only removed, reordered or layout changes) | `layout` | ignored |
| Text changed more than `DRASTIC_TEXT` **and** tag structure changed more than `DRASTIC_DOM` | `drastic` | cheap model: redesign, or did a tracked fact change? |
| New lines, but none match the domain's `keywords` | `no_keywords` | ignored |
| Otherwise | `normal` | cheap-model triage on the new lines only |

* **Cleaning** collapses whitespace and strips text matching `IGNORE_PATTERNS` (date-only lines, read times,
  copyright lines, cookie banners).
* **New lines** are lines of the new copy that appear nowhere in the old one, so moved lines don't count.
* **Text ratio** is `1 − difflib` similarity over the cleaned lines (0 = identical, 1 = nothing in common).
* **DOM ratio** is `1 − Jaccard` similarity of the pages' tag paths (`html>body>main>table>tr`), a cheap fingerprint of
  the page template.

In [ ]:
import difflib
import hashlib
import json
import re
from pathlib import Path


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def _snapshot_path(domain: str, name: str, ext: str = "txt") -> Path:
    return Path(SNAPSHOT_DIR) / domain / f"{_slug(name)}.{ext}"


def _read(path: Path) -> str | None:
    return path.read_text() if path.exists() else None


def load_snapshot(domain: str, name: str) -> str | None:
    return _read(_snapshot_path(domain, name))


def load_snapshot_html(domain: str, name: str) -> str | None:
    return _read(_snapshot_path(domain, name, "html"))


def load_meta(domain: str, name: str) -> dict:
    raw = _read(_snapshot_path(domain, name, "meta.json"))
    return json.loads(raw) if raw else {}


def load_baseline(domain: str, name: str) -> str | None:
    return _read(Path(BASELINE_DIR) / domain / f"{_slug(name)}.md")


def save_snapshot(domain: str, name: str, text: str, page: Fetched) -> None:
    files = {"txt": text, "html": page.html,
             "meta.json": json.dumps({"etag": page.etag, "last_modified": page.last_modified})}
    for ext, content in files.items():
        path = _snapshot_path(domain, name, ext)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content)


def _squash(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip().lower()


def clean_for_diff(text: str) -> str:
    """Collapse whitespace and strip volatile text (IGNORE_PATTERNS) so it can't register as a change."""
    lines = []
    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        for pattern in IGNORE_PATTERNS:
            line = re.sub(pattern, "", line, flags=re.IGNORECASE).strip()
        if line:
            lines.append(line)
    return "\n".join(lines)


def diff_text(old: str, new: str) -> str:
    lines = difflib.unified_diff(old.splitlines(), new.splitlines(), "before", "after", n=2, lineterm="")
    return "\n".join(lines)


_NON_CONTENT = "//head|//script|//style|//noscript|//template|//svg"


def tag_paths(html: str | None) -> set[str]:
    """Distinct tag chains such as 'html>body>main>table>tr': a cheap fingerprint of the page template."""
    if not html:
        return set()
    try:
        root = lxml.html.fromstring(html)
    except (lxml.etree.ParserError, ValueError):
        return set()
    for el in root.xpath(_NON_CONTENT):
        el.drop_tree()
    return {">".join([a.tag for a in reversed(list(el.iterancestors()))] + [el.tag])
            for el in root.iter() if isinstance(el.tag, str)}


def dom_change_ratio(old_html: str | None, new_html: str | None) -> float:
    old, new = tag_paths(old_html), tag_paths(new_html)
    if not old or not new:
        return 0.0   # nothing to compare (e.g. a text-only seeded baseline): treat structure as unchanged
    return 1 - len(old & new) / len(old | new)


@dataclass
class Signals:
    old_clean: str
    new_clean: str
    added: list[str]       # lines in the new copy that appear nowhere in the old one
    removed: list[str]     # lines in the old copy that appear nowhere in the new one
    text_change: float     # 0 = identical lines, 1 = nothing in common
    dom_change: float      # 0 = same tag structure, 1 = nothing in common
    length_ratio: float    # new cleaned length / old cleaned length

    def summary(self) -> dict:
        return {"added_lines": len(self.added), "removed_lines": len(self.removed),
                "text_change": round(self.text_change, 3), "dom_change": round(self.dom_change, 3),
                "length_ratio": round(self.length_ratio, 3)}


def _hash(text: str) -> bytes:
    return hashlib.sha256(text.encode()).digest()


def change_signals(old_text: str, new_text: str, old_html: str | None, new_html: str | None) -> Signals:
    old_clean, new_clean = clean_for_diff(old_text), clean_for_diff(new_text)
    if _hash(old_clean) == _hash(new_clean):   # the common case: skip the diffs entirely
        return Signals(old_clean, new_clean, [], [], 0.0, 0.0, 1.0)
    old_lines, new_lines = old_clean.splitlines(), new_clean.splitlines()
    old_set, new_set = {_squash(l) for l in old_lines}, {_squash(l) for l in new_lines}
    return Signals(
        old_clean, new_clean,
        added=[l for l in new_lines if _squash(l) not in old_set],
        removed=[l for l in old_lines if _squash(l) not in new_set],
        text_change=1 - difflib.SequenceMatcher(None, old_lines, new_lines).ratio(),
        dom_change=dom_change_ratio(old_html, new_html),
        length_ratio=len(new_clean) / max(len(old_clean), 1),
    )


def classify_change(sig: Signals, keywords: str) -> str:
    """none | broken | layout | drastic | no_keywords | normal (see the table above)."""
    if _hash(sig.old_clean) == _hash(sig.new_clean):
        return "none"
    if sig.length_ratio < BROKEN_RATIO:
        return "broken"
    if not sig.added:
        return "layout"
    if sig.text_change > DRASTIC_TEXT and sig.dom_change > DRASTIC_DOM:
        return "drastic"
    if not re.search(keywords, "\n".join(sig.added), re.IGNORECASE):
        return "no_keywords"
    return "normal"

### Cleaning example

What `clean_for_diff` removes, first on a made-up snippet, then on a real seeded baseline. The cleaned real file is
also written to `output/` so you can open it next to the original.

In [ ]:
sample = """Model deprecations
Last updated: October 28, 2026
   Example Model 1      retires on   February 19, 2027
5 min read
We use cookies to improve your experience. Accept all cookies
© 2026 Example AI Inc. All rights reserved."""
print("BEFORE\n------\n" + sample + "\n\nAFTER\n-----\n" + clean_for_diff(sample))

real_name = "Anthropic model deprecations"
real = load_baseline("models", real_name) or ""
cleaned = clean_for_diff(real)
kept = {_squash(l) for l in cleaned.splitlines()}
dropped = [l.strip() for l in real.splitlines() if l.strip() and _squash(l) not in kept]
print(f"\n{real_name} baseline: {len(real):,} → {len(cleaned):,} chars; "
      f"{len(dropped)} lines removed or trimmed{':' if dropped else ''}")
for line in dropped[:20]:
    print("  -", line)

cleaned_path = Path(OUTPUT_DIR) / f"cleaned-{_slug(real_name)}.txt"
cleaned_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_path.write_text(cleaned)
print(f"\ncleaned file: {cleaned_path}  (original: {Path(BASELINE_DIR) / 'models' / (_slug(real_name) + '.md')})")

### Check examples

Made-up before/after pairs run through `change_signals` + `classify_change`, one per outcome.

In [ ]:
import pandas as pd

BASE_TEXT = ("Model deprecations\nExample Model 1 retires on 2027-02-19\n"
             "Example Model 2 retires on 2027-06-01\nUse the migration guide to upgrade.")
BASE_HTML = "<html><body><main><h1>x</h1><table><tr><td>x</td></tr></table><p>x</p></main></body></html>"
REDESIGN_HTML = "<html><body><div><section><ul><li>x</li></ul></section><footer>x</footer></div></body></html>"
lines = BASE_TEXT.splitlines()

examples = {
    "whitespace only": (BASE_TEXT.replace(" ", "   "), BASE_HTML),
    "'Last updated' date added": ("Last updated: November 4, 2026\n" + BASE_TEXT, BASE_HTML),
    "layout tweak, same text": (BASE_TEXT, BASE_HTML.replace("<main>", "<main><div>").replace("</main>", "</div></main>")),
    "lines reordered": ("\n".join([lines[0], lines[2], lines[1], lines[3]]), BASE_HTML),
    "extraction broke": (lines[0], BASE_HTML),
    "redesign": ("Deprecated models\nExample Model 1: retirement date 2027-02-19\n"
                 "Example Model 2: retirement date 2027-06-01\nSee the upgrade guide.", REDESIGN_HTML),
    "new line, no keywords": (BASE_TEXT + "\nJoin our community forum.", BASE_HTML),
    "new deprecation row": (BASE_TEXT + "\nExample Model 3 retires on 2027-09-01", BASE_HTML),
}
rows = []
for label, (text, html) in examples.items():
    sig = change_signals(BASE_TEXT, text, BASE_HTML, html)
    rows.append({"example": label, "outcome": classify_change(sig, r"retir|deprecat|pric"), **sig.summary()})
pd.DataFrame(rows)

## LLM wrapper (Gemini on Vertex AI)

One function, `llm_json(prompt, schema, model, step)`, returns a validated pydantic object: `google-genai` with
`vertexai=True` and `response_schema`. `llm_text` returns free-form text for the digest.

Every call is appended to `LLM_CALLS` with its latency, token counts and estimated cost (`MODEL_PRICES`).
`with metered(domain, source):` attributes the calls made inside it to that source.

In [ ]:
from contextlib import contextmanager
from functools import cache

from google.genai import types

SYSTEM = ("You analyze changes to web pages for a monitoring service. Use only facts present in the input; "
          "never infer dates, prices or model names that are not written there.")

LLM_CALLS: list[dict] = []                 # one row per model call
_SCOPE = {"domain": "", "source": ""}      # who the current calls are for (set by `metered`)


@contextmanager
def metered(domain: str, source: str):
    """Attribute the model calls made inside this block to a domain and source."""
    previous = dict(_SCOPE)
    _SCOPE.update(domain=domain, source=source)
    try:
        yield
    finally:
        _SCOPE.update(previous)


def _record(model: str, step: str, started: float, resp) -> None:
    usage = resp.usage_metadata
    tokens_in = (usage.prompt_token_count or 0) if usage else 0
    # Gemini 2.5 thinking tokens are billed as output
    tokens_out = ((usage.candidates_token_count or 0) + (usage.thoughts_token_count or 0)) if usage else 0
    price_in, price_out = MODEL_PRICES.get(model, (0.0, 0.0))
    LLM_CALLS.append({**_SCOPE, "step": step, "model": model,
                      "latency_s": time.perf_counter() - started,
                      "tokens_in": tokens_in, "tokens_out": tokens_out,
                      "cost_usd": (tokens_in * price_in + tokens_out * price_out) / 1e6})


@cache
def _gemini():
    from google import genai
    return genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)


def llm_json(prompt: str, schema: type[BaseModel], model: str, step: str) -> BaseModel | None:
    started = time.perf_counter()
    resp = _gemini().models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM,
                                           response_mime_type="application/json",
                                           response_schema=schema),
    )
    _record(model, step, started, resp)
    if resp.parsed is None:
        print(f"  {model} returned no parseable output; skipping")
        return None
    return resp.parsed


def llm_text(prompt: str, model: str, step: str) -> str:
    started = time.perf_counter()
    resp = _gemini().models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM),
    )
    _record(model, step, started, resp)
    return resp.text

## Triage + extract (with the quote guard)

Only pages the checks can't settle reach a model. `triage` sees just the new lines (plus a few removed lines for
context). `redesign_check` handles the `drastic` case: it compares removed and added lines and says whether a tracked
fact changed or only the wording and layout did. Both verdicts are logged to `TRIAGE_LOG` with the check signals, as
training data for a future classifier. Extraction turns the diff into structured events, and any event whose quote
isn't found in the new page text is dropped.

In [ ]:
from datetime import datetime, timezone


def _clip(text: str, limit: int = MAX_DIFF_CHARS) -> str:
    if len(text) <= limit:
        return text
    print(f"  input is {len(text):,} chars; sending the first {limit:,} (raise MAX_DIFF_CHARS to send more)")
    return text[:limit]


def triage(sig: Signals, source: dict, focus: str) -> Triage | None:
    added = _clip("\n".join(sig.added))
    removed = _clip("\n".join(sig.removed), MAX_DIFF_CHARS // 4)
    prompt = (f"Source: {source['name']} ({source['url']})\nWe track: {focus}\n\n"
              f"Lines added since the last check:\n<added>\n{added}\n</added>\n\n"
              f"Lines removed (context only):\n<removed>\n{removed}\n</removed>\n\n"
              "Is this a meaningful change for what we track?")
    return llm_json(prompt, Triage, TRIAGE_MODEL, step="triage")


def redesign_check(sig: Signals, source: dict, focus: str) -> Triage | None:
    removed = _clip("\n".join(sig.removed), MAX_DIFF_CHARS // 2)
    added = _clip("\n".join(sig.added), MAX_DIFF_CHARS // 2)
    prompt = (f"Source: {source['name']} ({source['url']})\nWe track: {focus}\n\n"
              f"Both the wording ({sig.text_change:.0%} changed) and the page structure ({sig.dom_change:.0%} changed) "
              "of this page changed a lot, which is often a site redesign.\n\n"
              f"Old lines:\n<removed>\n{removed}\n</removed>\n\nNew lines:\n<added>\n{added}\n</added>\n\n"
              "Set meaningful=true only if a tracked fact (a model, date, price, limit, rule or obligation) was added, "
              "removed or changed, not if the same facts were just reworded or moved.")
    return llm_json(prompt, Triage, TRIAGE_MODEL, step="redesign")


def log_verdict(domain: str, source: dict, step: str, verdict: Triage | None, sig: Signals) -> None:
    """Append a model verdict and its signals to TRIAGE_LOG: training data for a future classifier."""
    path = Path(TRIAGE_LOG)
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {"time": datetime.now(timezone.utc).isoformat(timespec="seconds"), "domain": domain,
              "source": source["name"], "step": step, **sig.summary(),
              "verdict": verdict.model_dump() if verdict else None,
              "added_text": "\n".join(sig.added)[:MAX_DIFF_CHARS]}
    with path.open("a") as f:
        f.write(json.dumps(record) + "\n")


def extract(diff: str, new_text: str, source: dict, domain: "Domain") -> list[BaseModel]:
    prompt = (f"Source: {source['name']} — {source['url']}\nProvider/publisher: {source['provider']}\n"
              f"We track: {domain.focus}\n\nUnified diff of the page text:\n<diff>\n{_clip(diff)}\n</diff>\n\n"
              "List one event per distinct change shown by the added (+) lines. For `quote`, copy the "
              "relevant text exactly as it appears in the + lines. Set source_url to the source URL above. "
              "Return an empty list if nothing relevant changed.")
    result = llm_json(prompt, domain.events_schema, EXTRACT_MODEL, step="extract")
    events = result.events if result else []
    return [e for e in events if quote_is_verbatim(e, new_text, source)]


def quote_is_verbatim(event: BaseModel, page_text: str, source: dict) -> bool:
    event.source_url = source["url"]   # never trust the model with the citation itself
    if _squash(event.quote) in _squash(page_text):
        return True
    print(f"  dropped event (quote not found in page): {event.summary[:80]!r}")
    return False

In [ ]:
from typing import Callable


@dataclass
class Domain:
    name: str
    sources: list[dict]
    focus: str
    keywords: str          # regex; new lines with no match are ignored without a model call. Keep it permissive.
    events_schema: type[BaseModel]
    dedupe_key: Callable[[BaseModel], tuple]
    digest_instructions: str


MODELS = Domain(
    name="models",
    sources=MODEL_SOURCES,
    focus="AI model launches, deprecations and retirement dates, pricing, API/parameter changes, rate limits, capabilities",
    keywords=(r"deprecat|retir|sunset|discontinu|shut ?down|end of life|launch|releas|introduc|now available"
              r"|general(ly)? availab|\bGA\b|preview|beta|experimental|pric|\$\s?\d|per (1M|million)|tokens?"
              r"|rate limit|quota|context (window|length)|parameter|endpoint|\bAPI\b|support"
              r"|\b(claude|gemini|gemma|imagen|veo|gpt|o\d|mistral|codestral|magistral|devstral|pixtral|ministral)\b"),
    events_schema=ChangeEvents,
    dedupe_key=lambda e: (e.provider.lower(), (e.model or "").lower(), e.change_type),
    digest_instructions="Group by provider. Put breaking changes (retirements, removed parameters, price increases) first.",
)

## The pipeline

`run(domain)` runs `check_source` on every page: fetch → normalize → change checks → (triage) → extract, then
dedupes. Each status row says which check decided (`decided_by`: `etag`, `hash`, `broken`, `layout`, `keywords` or
`llm`), how long each stage took, and the model calls made for that source. The first run diffs against the seeded
baselines; run it again and every source should report no change with no model calls.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display


def dedupe(events: list[BaseModel], key: Callable[[BaseModel], tuple]) -> list[BaseModel]:
    merged: dict[tuple, BaseModel] = {}
    for e in events:
        merged.setdefault(key(e), e)   # first source wins; production would merge citations
    return list(merged.values())


def check_source(domain: Domain, src: dict) -> tuple[dict, list[BaseModel]]:
    """Fetch one page, run the change checks, and call a model only when the checks can't decide."""
    name = src["name"]
    row = {"source": name, "status": "", "decided_by": "", "events": 0, "raw_changed": False,
           "fetch_s": 0.0, "parse_s": 0.0, "checks_s": 0.0}
    try:
        page = fetch(src["url"], load_meta(domain.name, name))
    except httpx.HTTPStatusError as e:
        hint = " (bot-protected: needs Playwright or an Open Terms Archive feed)" if e.response.status_code == 403 else ""
        return {**row, "status": f"HTTP {e.response.status_code}{hint}", "decided_by": "error"}, []
    except httpx.HTTPError as e:
        return {**row, "status": f"fetch error: {e.__class__.__name__}", "decided_by": "error"}, []
    row["fetch_s"] = page.seconds
    if page.html is None:
        return {**row, "status": "no change (304 Not Modified)", "decided_by": "etag"}, []

    started = time.perf_counter()
    new_text = normalize(page.html)
    row["parse_s"] = time.perf_counter() - started
    if not new_text:
        return {**row, "status": "empty after normalize (JS-rendered? try Playwright)", "decided_by": "error"}, []

    old_text, old_html, vs = load_snapshot(domain.name, name), load_snapshot_html(domain.name, name), ""
    if old_text is None:
        old_text, vs = load_baseline(domain.name, name), " vs seeded baseline"
    if old_text is None:
        save_snapshot(domain.name, name, new_text, page)
        return {**row, "status": "first snapshot saved", "decided_by": "first_snapshot"}, []

    started = time.perf_counter()
    sig = change_signals(old_text, new_text, old_html, page.html)
    outcome = classify_change(sig, domain.keywords)
    row["checks_s"] = time.perf_counter() - started
    row["raw_changed"] = old_text != new_text   # what the old pipeline would have sent to triage
    if outcome != "broken":   # keep the last good copy until extraction works again
        save_snapshot(domain.name, name, new_text, page)

    if outcome == "none":
        return {**row, "status": f"no change{vs}", "decided_by": "hash"}, []
    if outcome == "broken":
        return {**row, "status": f"extraction broke? text is {sig.length_ratio:.0%} of the previous copy; "
                                 "snapshot kept", "decided_by": "broken"}, []
    if outcome == "layout":
        return {**row, "status": f"ignored{vs}: no new lines ({len(sig.removed)} removed or reordered)",
                "decided_by": "layout"}, []
    if outcome == "no_keywords":
        return {**row, "status": f"ignored{vs}: {len(sig.added)} new lines, no tracked keywords",
                "decided_by": "keywords"}, []

    check = redesign_check if outcome == "drastic" else triage
    verdict = check(sig, src, domain.focus)
    log_verdict(domain.name, src, check.__name__, verdict, sig)
    row["decided_by"] = "llm"
    if not verdict or not verdict.meaningful:
        label = "redesign, no tracked facts changed; rebaselined" if outcome == "drastic" else "ignored"
        return {**row, "status": f"{label}{vs}: {verdict.reason if verdict else 'no verdict'}"}, []
    found = extract(diff_text(sig.old_clean, sig.new_clean), new_text, src, domain)
    return {**row, "status": f"changed ({verdict.severity}){vs}", "events": len(found)}, found


def run(domain: Domain) -> tuple[list[BaseModel], pd.DataFrame]:
    LLM_CALLS[:] = [c for c in LLM_CALLS if c["domain"] != domain.name]   # re-running a domain replaces its numbers
    events, rows = [], []
    for src in domain.sources:
        with metered(domain.name, src["name"]):
            row, found = check_source(domain, src)
        calls = [c for c in LLM_CALLS if c["domain"] == domain.name and c["source"] == src["name"]]
        rows.append({**row, "llm_calls": len(calls),
                     "llm_s": sum(c["latency_s"] for c in calls),
                     "tokens_in": sum(c["tokens_in"] for c in calls),
                     "tokens_out": sum(c["tokens_out"] for c in calls),
                     "cost_usd": sum(c["cost_usd"] for c in calls)})
        events.extend(found)
    status = pd.DataFrame(rows).round({"fetch_s": 2, "parse_s": 2, "checks_s": 3, "llm_s": 2, "cost_usd": 5})
    return dedupe(events, domain.dedupe_key), status


model_events, model_status = run(MODELS)
model_status

## Models digest

The digest is written **only from the extracted events** (never from raw pages), and every bullet links its source. With no events the digest is empty and no LLM call is made.

In [ ]:
def digest(domain: Domain, events: list[BaseModel], title: str) -> str:
    if not events:
        return ""
    payload = json.dumps([e.model_dump() for e in events], indent=1)
    prompt = (f"Write a concise Markdown digest titled '## {title}' from these events:\n<events>\n{payload}\n</events>\n\n"
              f"{domain.digest_instructions} One bullet per event, ending with a Markdown link to its source_url. "
              "Do not add facts that are not in the events.")
    with metered(domain.name, "<digest>"):
        return llm_text(prompt, DIGEST_MODEL, step="digest")


models_digest = digest(MODELS, model_events, "AI model changes")
display(Markdown(models_digest or "_No model changes to report._"))

## Legal subagent

Same `run()` and `digest()` — only the sources, schema, focus and digest instructions differ.

In [ ]:
LEGAL = Domain(
    name="legal",
    sources=LEGAL_SOURCES,
    focus=("Legal and regulatory developments affecting companies that provide LLM services: terms of service and "
           "usage-policy changes, EU AI Act / GPAI obligations and guidance, US federal and state AI rules, "
           "enforcement actions, and lawsuits (copyright, privacy, consumer protection)"),
    keywords=(r"\b(shall|must|may not|prohibit|permit|allow|restrict|requir|obligat|comply|complian)"
              r"|effective|enter(s|ed)? into (force|application)|apply from|deadline|amend|updat|revis|replac"
              r"|regulation|directive|\bact\b|article \d+|annex|guideline|guidance|code of practice|template"
              r"|court|lawsuit|complaint|settlement|fine|penalt|enforce|consultation|signator"),
    events_schema=LegalEvents,
    dedupe_key=lambda e: (e.jurisdiction, _squash(e.title)),
    digest_instructions=("Group by jurisdiction, then litigation and terms-of-service changes. For each item, "
                         "state the practical impact on LLM providers. This is monitoring, not legal advice."),
)

legal_events, legal_status = run(LEGAL)
display(legal_status)
legal_digest = digest(LEGAL, legal_events, "Legal & regulatory updates for LLM providers")
display(Markdown(legal_digest or "_No legal changes to report._"))

## Cost & latency

Where the time and money went, per source and per model, and what the change checks saved. The old pipeline sent
every page whose text differed at all to triage, so "avoided" counts pages whose raw text changed but were settled
without a model. The saving uses this run's measured mean triage latency and cost (falling back to
`FALLBACK_TRIAGE_*` when there were no triage calls). That is conservative, because the old triage used Flash on the
full diff, not Flash-Lite on the new lines.

In [ ]:
CALL_COLUMNS = ["domain", "source", "step", "model", "latency_s", "tokens_in", "tokens_out", "cost_usd"]
calls = pd.DataFrame(LLM_CALLS, columns=CALL_COLUMNS)
status = pd.concat([model_status.assign(domain="models"), legal_status.assign(domain="legal")], ignore_index=True)

print("Per source")
display(status[["domain", "source", "decided_by", "fetch_s", "parse_s", "checks_s",
                "llm_calls", "llm_s", "tokens_in", "tokens_out", "cost_usd"]])

print("Per model and step (digest calls included)")
display(calls.groupby(["model", "step"]).agg(
    calls=("model", "size"), mean_latency_s=("latency_s", "mean"), total_latency_s=("latency_s", "sum"),
    tokens_in=("tokens_in", "sum"), tokens_out=("tokens_out", "sum"), cost_usd=("cost_usd", "sum"),
).round({"mean_latency_s": 2, "total_latency_s": 2, "cost_usd": 5}))

triage_calls = calls[calls["step"].isin(["triage", "redesign"])]
if len(triage_calls):
    mean_s, mean_usd = triage_calls["latency_s"].mean(), triage_calls["cost_usd"].mean()
    basis = f"mean of {len(triage_calls)} measured triage calls"
else:
    mean_s, mean_usd, basis = FALLBACK_TRIAGE_SECONDS, FALLBACK_TRIAGE_USD, "fallback constants from the config cell"
avoided = status[status["raw_changed"] & (status["decided_by"] != "llm")]
print(f"Triage calls avoided: {len(avoided)} of {int(status['raw_changed'].sum())} pages whose text changed")
print(f"Estimated saving: {len(avoided) * mean_s:.1f}s of model latency and ${len(avoided) * mean_usd:.5f} ({basis}); "
      f"the checks themselves took {status['checks_s'].sum():.3f}s in total")
display(avoided[["domain", "source", "decided_by", "status"]])

## Save digest

The combined digest (with the charts above) is written to `output/digest-<date>.md` and rendered below.
If neither domain has anything to report, no file is written and `digest_path` is `None`.
Email delivery comes next.

In [ ]:
def save_digest(text: str) -> Path:
    path = Path(OUTPUT_DIR) / f"digest-{date.today().isoformat()}.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    print(f"saved {len(text):,} chars to {path}")
    return path


sections = [d for d in (models_digest, legal_digest) if d]
if sections:
    digest_path = save_digest("\n\n".join(sections))
    # Render the saved file; image links are rewritten to resolve from the notebook's directory
    display(Markdown(digest_path.read_text().replace("](chart-", f"]({OUTPUT_DIR}/chart-")))
else:
    digest_path = None
    print("No changes to report; no digest written.")